# RAG Pipeline: 

## 1. Document Preparation

we will deal with unstructured data (PDF file) and will see how we can preprocess the text and convert to the dataset we will use

first we need to parse the pdf file into Markdown file so we can use it later in text preprocessing and chunking

In [1]:
import pymupdf


def pdf_to_markdown(pdf_path, markdown_path):
    
    """Extract the text from a PDF and save it as a Markdown file."""

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(r"D:\Ai Track\RAG\Stranger_Things.pdf", "output.md")


Markdown file created: output.md


## 2. Chunking

now we have one Markdown file containing all the pages of the pdf file , we need to preprocess the page content and chunk them 

In [ ]:
from pathlib import Path
import re

INPUT_FILE = Path("output.md")
OUTPUT_FOLDER = Path("dataset")

BOOK_RANGES = [
    ("part_1", 4, 30),
    ("part_2", 31, 60),
    ("part_3", 61, 90),
    ("part_4", 91, 118),
    ("part_5", 119, 149),
    ("part_6", 150, 190),
    ("part_7", 191, 219),
]

def get_part(page_number):
    
    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

    return None
def clean_text(text):

    text = re.sub(r'\s+', ' ', text)
    text = text.replace(r'[\r\n]+', '\n').strip()

    return text   
def split_pages():

    text = INPUT_FILE.read_text(encoding="utf-8")

    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)

    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):

        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_part(page_number)

        if book_name:

            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )

            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")

            

split_pages()



----------------------------------------

## 3. Embeddings

We will now create an embedding for the content of every page.

For each page, we will keep three pieces of information as its payload:

- the book name
- the page number
- the page content

In [3]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

DATASET_FOLDER = Path("dataset")
MODEL_NAME = "intfloat/multilingual-e5-small"


def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }



files = sorted(DATASET_FOLDER.glob("*.md"))
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()



c:\Users\Mohammed Atta\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 7/7 [00:29<00:00,  4.25s/it]


In [ ]:
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os

load_dotenv(r"D:\Ai Track\RAG\project\.env")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")


client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )


points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 100

for start in range(0, len(points), batch_size):
    batch = points[start:start + batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch,
    )

print(f"Uploaded {len(points)} pages to Qdrant.")

ResponseHandlingException: The write operation timed out

In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

load_dotenv(r"D:\Ai Track\RAG\project\.env")

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60.0,  
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )

points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 32

for start in range(0, len(points), batch_size):
    batch = points[start : start + batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch,
        wait=True,
    )
    print(f"Uploaded batch {start // batch_size + 1} / {(len(points) + batch_size - 1) // batch_size}")

print(f"\nSuccessfully uploaded all {len(points)} pages to Qdrant!")

Uploaded batch 1 / 7
Uploaded batch 2 / 7
Uploaded batch 3 / 7
Uploaded batch 4 / 7
Uploaded batch 5 / 7
Uploaded batch 6 / 7
Uploaded batch 7 / 7

Successfully uploaded all 216 pages to Qdrant!


#### **Here the preparation of the data is done , you can now switch to the `rag_api.py` to continue the project , the rest of the cells below are for just demonestrating the rest of the pipeline**

-----------------------------------

## 4. Query Router



In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(r"D:\Ai Track\RAG\project\.env")

query = input("Ask a question: ")

router_llm = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    google_api_key=os.getenv("GROQ_API_KEY")
)

SYSTEM_PROMPT = """You classify messages for a Stranger Things book search system.
Return exactly one of these labels and nothing else:
retrieve
chitchat
off-topic

Rules:
- retrieve: Any questions about Stranger Things books, characters (Eleven, Mike, Hopper, Will, etc.), upside down, monsters, places, or events.
- chitchat: Greetings (hi, hello), thanks, or casual conversation.
- off-topic: Anything completely unrelated to Stranger Things."""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]

response = router_llm.invoke(router_messages).content

if isinstance(response, list):
    raw_content = response[0] if response else ""
else:
    raw_content = response

route = str(raw_content).strip().lower()
route = route.splitlines()[0].strip(" `.,:'\"")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)

Route: off-topic


## 4. Retrieval

We can now search the vector database.

The query is converted into an embedding using the same model. Qdrant compares it with the stored page embeddings and returns the most similar pages.

In [43]:
route = "retrieve" 

if route == "retrieve":
    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(
        collection_name="stranger_things",
        query=query_vector,
        limit=top_k,
    ).points

    context = ""
    for result in results:
        page = result.payload
        context += (
            f"Book: {page['book_name']}\n"
            f"Page: {page['page_number']}\n"
            f"Content: {page['content']}\n\n"
        )

        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"])
        print("-" * 80)
else:
    print("No database search needed.")

Score: 0.81570816
Book: part_1
Page: 17
Content: As Chief Hopper reminds us, “Mornings are for coffee and contemplation.” We couldn’t agree more.
--------------------------------------------------------------------------------
Score: 0.8134444
Book: part_1
Page: 4
Content: Popular Culture and Philosophy® Series Editor: George A. Reisch VOLUME 1 Seinfeld and Philosophy: A Book about Everything and Nothing (2000) VOLUME 2 The Simpsons and Philosophy: The D’oh! of Homer (2001) VOLUME 3 The Matrix and Philosophy: Welcome to the Desert of the Real (2002) VOLUME 4 Buffy the Vampire Slayer and Philosophy: Fear and Trembling in Sunnydale (2003) VOLUME 9 Harry Potter and Philosophy: If Aristotle Ran Hogwarts (2004) VOLUME 12 Star Wars and Philosophy: More Powerful than You Can Possibly Imagine (2005) VOLUME 13 Superheroes and Philosophy: Truth, Justice, and the Socratic Way (2005) VOLUME 17 Bob Dylan and Philosophy: It’s Alright Ma (I’m Only Thinking) (2006) VOLUME 19 Monty Python and Philosoph

## 5. Keyword Search

Semantic search finds similar meaning using embeddings. Keyword search looks for the exact words in the page content.

For now, we will search the pages already loaded in `pages`. This is a simple keyword search; BM25 can be added later as a stronger keyword-ranking method.


In [40]:
if route == "retrieve":
    keyword_query = "LUCAS"
    top_k = 3

    keywords = keyword_query.lower().split()
    keyword_results = []

    for page in pages:
        content = page["content"].lower()
        score = sum(content.count(keyword) for keyword in keywords)

        if score > 0:
            keyword_results.append({
                "score": score,
                "book_name": page["book_name"],
                "page_number": page["page_number"],
                "content": page["content"],
            })

    keyword_results.sort(key=lambda result: result["score"], reverse=True)

    for result in keyword_results[:top_k]:
        print("Keyword score:", result["score"])
        print("Book:", result["book_name"])
        print("Page:", result["page_number"])
        print("Content:", result["content"])
        print("-" * 80)


Keyword score: 6
Book: part_6
Page: 178
Content: actions and morality in life. Aristotle’s virtues of honesty and courage and all that? Nietzsche thinks we’re deluding ourselves in thinking that we can ever reach such lofty goals. Instead, we ought to spend our time trying to live our lives as greatly as possible. In terms that even Mad Max would approve of, Nietzsche thinks we should do everything we can to not be a total wasteoid. So, what does this have to do with friendship? Nietzsche’s concept of friendship is based in this “natural state” of humanity, in which our friends are those people who challenge us to live our best possible lives. The way that Nietzsche suggests others should challenge us to live our best lives is to be those people who actually challenge us to do so. Through conflict. Probably Nietzsche’s most famous quote is the incredible: “Whatever doesn’t kill me makes me stronger.” This is the sort of thing he’s talking about there. Your friends will make you stronge

## 6. Generation

the final step where a Large Language Model (LLM) creates a final, natural language response using both the user's original query and the context retrieved from the QDRANT DB

In [ ]:
if route == "retrieve":
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import SystemMessage, HumanMessage

    gemini_llm = ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL"),
        api_key=os.getenv("GEMINI_API_KEY"),
    )

    messages = [
        SystemMessage(
            content="Answer only from the provided pages. If the answer is not there, say you do not know. Keep the answer concise."
        ),
        HumanMessage(
            content=f"Context:\n{context}\nQuestion:\n{query}"
        ),
    ]

    response = gemini_llm.invoke(messages)
    print("\nAnswer:")
    print(response.text)


Answer:
Based on the provided text, Hopper (Chief Hopper) is a father figure to Eleven whose primary goal is to protect her and give her as normal a childhood as possible.
